# Classification automatique de décisions judiciaires françaises par type d'infraction
**Awa Traoré DIOP | Projet NLP personnel | Data Science**

## Vue d'ensemble du projet

Ce projet combine deux sources de données publiques françaises :

- **Source 1 — Labels** : Les 107 catégories d'infractions de l'ONDRP (data.gouv.fr)
- **Source 2 — Textes** : Les décisions de justice de la Cour de cassation via l'API Judilibre

**Objectif** : Entraîner un modèle NLP (CamemBERT) capable de classifier automatiquement
une décision de justice par type d'infraction, sans intervention humaine.

**Application directe** : détection de fraudes (DGDDI), analyse de dossiers médicaux (santé)

---
## Prérequis avant de démarrer

**1. Clé API Judilibre** (gratuit, 5 minutes) :
- sur https://piste.api.gouv.fr
- Création d'un compte gratuit
- Souscription à l'API Judilibre
- Récupération de la clé API

**2. Activation du GPU sur Colab** :
- Menu Exécution > Modifier le type d'exécution > GPU T4

---
> **Comment lire ce notebook** : chaque cellule commence par un commentaire qui explique
> *pourquoi* on fait ça, pas seulement *comment*.

---
## ⚙️ Phase 0 — Installation & Configuration
---

In [ ]:
# Installation de toutes les dépendances
# transformers : modèles NLP pré-entraînés (CamemBERT)
# spacy        : extraction d'entités nommées
# requests      : appels à l'API Judilibre
# openpyxl      : lecture du fichier xlsx des infractions

!pip install transformers datasets spacy torch requests openpyxl scikit-learn -q
!python -m spacy download fr_core_news_md -q
print('Installation terminee !')

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re, json, time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

In [ ]:
API_KEY = 'API_KEY'

# URL de base de l'API Judilibre
JUDILIBRE_URL = 'https://api.piste.gouv.fr/cassation/judilibre/v1.0'

# Headers requis pour chaque appel API
HEADERS = {
    'KeyId': API_KEY,
    'Accept': 'application/json'
}

# Test de connexion
resp = requests.get(f'{JUDILIBRE_URL}/healthcheck', headers=HEADERS)
if resp.status_code == 200:
    print('Connexion API Judilibre OK !')
else:
    print(f'Erreur : {resp.status_code} — verifie ta cle API')

---
## Phase 1 — Construction des labels depuis l'ONDRP
---

### Pourquoi cette étape ?

Les 107 catégories d'infractions de l'ONDRP sont notre référentiel de classification.
On va les télécharger, les nettoyer, puis les regrouper en grandes familles
pour avoir des classes équilibrées — condition essentielle pour un bon modèle NLP.

**Règle d'or** : trop de classes = modèle confus. On vise 5 à 8 catégories finales.

In [ ]:
# Téléchargement du fichier ONDRP depuis data.gouv.fr
# Ce fichier contient les 107 index d'infractions officiels de la police nationale
# Note : ce fichier date de 2014 mais les 107 catégories d'infractions
# (État 4001) sont un référentiel officiel stable depuis 1972.
# On l'utilise uniquement pour extraire les libellés des infractions,
# pas les volumes statistiques — qui eux sont dans les bases SSMSI 2024.

url_ondrp = 'https://www.data.gouv.fr/api/1/datasets/r/0dc8dca0-b608-49fc-9dbc-cfa7fd5a221a'
print('Téléchargement du fichier ONDRP...')

response = requests.get(url_ondrp)
with open('ondrp_infractions.xlsx', 'wb') as f:
    f.write(response.content)

# Lecture du fichier
# On explore d'abord toutes les feuilles pour comprendre la structure
xl = pd.ExcelFile('ondrp_infractions.xlsx')
print(f'Feuilles disponibles : {xl.sheet_names}')

In [ ]:
# Chargement et exploration de la structure
# On lit la première feuille pour voir ce qu'on a
df_raw = pd.read_excel('ondrp_infractions.xlsx', sheet_name=0, header=None)
print(f'Dimensions : {df_raw.shape}')
print()
df_raw.head(20)

In [ ]:
# Extraction des libellés d'infractions
# On cherche la colonne contenant les noms des 107 infractions
# Adapte l'index de colonne selon ce que tu vois dans la cellule précédente

# Exploration : affiche les valeurs uniques non-nulles de chaque colonne
for col in df_raw.columns[:5]:
    vals = df_raw[col].dropna().unique()[:5]
    print(f'Colonne {col} : {vals}')
    print()

In [ ]:
# Identifier la bonne colonne, extrais les infractions
COL_INFRACTIONS = 2  # <- colonne avec les infractions

infractions_raw = df_raw[COL_INFRACTIONS].dropna().tolist()
# Filtre les lignes qui ressemblent à des libellés (longueur > 5 caractères)
infractions = [str(i).strip() for i in infractions_raw if len(str(i).strip()) > 5]

print(f'Nombre d infractions extraites : {len(infractions)}')
print()
for i, inf in enumerate(infractions[:20], 1):
    print(f'{i:3}. {inf}')

In [ ]:
# Regroupement en grandes catégories thématiques
# On passe de 107 infractions détaillées à 6 grandes familles

# Dictionnaire de mapping : mots-clés -> catégorie
CATEGORIES = {
    'Atteintes aux personnes': [
        'homicide', 'meurtre', 'coups', 'blessures', 'violence', 'viol',
        'agression', 'menace', 'harcelement', 'harcèlement', 'personne'
    ],
    'Atteintes aux biens': [
        'vol', 'cambriolage', 'escroquerie', 'fraude', 'recel', 'destruction',
        'degradation', 'dégradation', 'bien', 'propriete', 'propriété'
    ],
    'Infractions economiques': [
        'blanchiment', 'corruption', 'trafic', 'contrefacon', 'contrefaçon',
        'faux', 'abus', 'detournement', 'détournement', 'douane'
    ],
    'Infractions aux personnes vulnerables': [
        'enfant', 'mineur', 'famille', 'abandon', 'maltraitance', 'proxenetisme',
        'proxénétisme', 'traite'
    ],
    'Infractions a l ordre public': [
        'stupefiant', 'stupéfiant', 'arme', 'terrorisme', 'association',
        'rebellion', 'rébellion', 'outrage'
    ],
    'Infractions routieres': [
        'conduite', 'alcool', 'vehicule', 'véhicule', 'route', 'accident'
    ]
}

def categoriser(infraction):
    """Assigne une catégorie à une infraction selon les mots-clés"""
    inf_lower = infraction.lower()
    for categorie, mots_cles in CATEGORIES.items():
        if any(mot in inf_lower for mot in mots_cles):
            return categorie
    return 'Autres infractions'

# Application du mapping
df_labels = pd.DataFrame({'infraction': infractions})
df_labels['categorie'] = df_labels['infraction'].apply(categoriser)

print('Distribution des categories :')
print(df_labels['categorie'].value_counts())

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
counts = df_labels['categorie'].value_counts()
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(counts)))
ax.barh(counts.index, counts.values, color=colors)
ax.set_title('Répartition des infractions par catégorie', fontsize=13, fontweight='bold')
ax.set_xlabel('Nombre d infractions')
for i, v in enumerate(counts.values):
    ax.text(v + 0.1, i, str(v), va='center')
plt.tight_layout()
plt.savefig('categories_infractions.png', dpi=150, bbox_inches='tight')
plt.show()

# Sauvegarde des labels
label2id = {label: idx for idx, label in enumerate(df_labels['categorie'].unique())}
id2label = {v: k for k, v in label2id.items()}
print(f'\nMapping labels : {label2id}')

---
## Phase 2 — Collecte des décisions via l'API Judilibre
---

### Pourquoi cette étape ?

On va interroger l'API Judilibre pour récupérer des décisions de justice réelles,
en utilisant les mots-clés de chaque catégorie comme requêtes de recherche.

**Stratégie** : pour chaque catégorie, on envoie une requête avec un mot-clé représentatif
et on récupère les décisions correspondantes. Le texte de la décision devient notre
donnée d'entraînement, et la catégorie devient le label.

**Limite à connaître** : l'API a des quotas — on va collecter 50 décisions par catégorie,
soit environ 300 décisions au total. C'est suffisant pour un projet démonstratif.

In [ ]:
def rechercher_decisions(mot_cle, batch_size=10, page=0):
    """
    Interroge l'API Judilibre avec un mot-clé
    Retourne une liste de décisions avec leur texte et métadonnées
    """
    params = {
        'query': mot_cle,
        'page_size': batch_size,
        'page': page,
        'resolve_references': 'false'
    }

    try:
        resp = requests.get(
            f'{JUDILIBRE_URL}/search',
            headers=HEADERS,
            params=params,
            timeout=15
        )
        if resp.status_code != 200:
            print(f'  Erreur {resp.status_code} pour : {mot_cle}')
            return []

        data = resp.json()
        resultats = data.get('results', [])
        decisions = []

        for r in resultats:
            texte = r.get('text', '') or r.get('summary', '')
            if texte and len(texte.split()) > 30:  # filtre textes trop courts
                decisions.append({
                    'id': r.get('id', ''),
                    'date': r.get('decision_date', ''),
                    'chambre': r.get('chamber', ''),
                    'texte': texte[:2000],  # on tronque à 2000 chars
                    'mot_cle': mot_cle
                })
        return decisions

    except Exception as e:
        print(f'  Exception pour {mot_cle} : {e}')
        return []


# Test rapide avant la collecte complète
test = rechercher_decisions('vol', batch_size=2)
if test:
    print(f'Test OK : {len(test)} décision(s) récupérée(s)')
    print(f'Extrait : {test[0]["texte"][:200]}...')
else:
    print('Test échoué — vérifie ta clé API')

In [ ]:
# Collecte complète : 50 décisions par catégorie
# On utilise les mots-clés les plus représentatifs de chaque catégorie

REQUETES_PAR_CATEGORIE = {
    'Atteintes aux personnes'          : ['violence', 'coups blessures', 'agression'],
    'Atteintes aux biens'              : ['vol', 'escroquerie', 'cambriolage'],
    'Infractions economiques'          : ['fraude', 'blanchiment', 'corruption'],
    'Infractions aux personnes vulnerables': ['mineur', 'abandon famille'],
    'Infractions a l ordre public'     : ['stupefiant', 'arme', 'terrorisme'],
    'Infractions routieres'            : ['conduite alcool', 'accident vehicule'],
}

NB_PAR_CATEGORIE = 50  # decisions par categorie
corpus = []

for categorie, requetes in REQUETES_PAR_CATEGORIE.items():
    print(f'\nCategorie : {categorie}')
    decisions_categorie = []

    for requete in requetes:
        if len(decisions_categorie) >= NB_PAR_CATEGORIE:
            break
        restant = NB_PAR_CATEGORIE - len(decisions_categorie)
        decisions = rechercher_decisions(requete, batch_size=min(restant, 20))
        for d in decisions:
            d['categorie'] = categorie
            d['label'] = label2id.get(categorie, 6)
        decisions_categorie.extend(decisions)
        time.sleep(0.5)  # respect des quotas API

    corpus.extend(decisions_categorie)
    print(f'  -> {len(decisions_categorie)} decisions collectees')

df_corpus = pd.DataFrame(corpus)
print(f'\nCorpus total : {len(df_corpus)} decisions')
print(df_corpus['categorie'].value_counts())

# Sauvegarde — important ! évite de refaire l'appel API
df_corpus.to_csv('corpus_judilibre.csv', index=False, encoding='utf-8')
print('\nCorpus sauvegarde dans corpus_judilibre.csv')

In [ ]:
# Exploration du corpus collecté
# On recharge depuis le fichier sauvegardé — bonne pratique
df_corpus = pd.read_csv('corpus_judilibre.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des classes
counts = df_corpus['categorie'].value_counts()
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(counts)))
axes[0].barh(counts.index, counts.values, color=colors)
axes[0].set_title('Decisions par categorie', fontsize=12, fontweight='bold')
for i, v in enumerate(counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center')

# Longueur des textes
df_corpus['nb_mots'] = df_corpus['texte'].apply(lambda x: len(str(x).split()))
axes[1].hist(df_corpus['nb_mots'], bins=30, color='#2E86AB', alpha=0.8)
axes[1].set_title('Longueur des decisions (mots)', fontsize=12, fontweight='bold')
axes[1].axvline(df_corpus['nb_mots'].median(), color='red', linestyle='--',
                label=f'Mediane : {df_corpus["nb_mots"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('exploration_corpus.png', dpi=150, bbox_inches='tight')
plt.show()

# Exemple de decision
print('\nExemple de decision :')
exemple = df_corpus.iloc[0]
print(f'Categorie : {exemple["categorie"]}')
print(f'Chambre   : {exemple["chambre"]}')
print(f'Date      : {exemple["date"]}')
print(f'Texte     : {str(exemple["texte"])[:300]}...')

---
## Phase 3 — Modélisation NLP avec CamemBERT
---

### Pourquoi CamemBERT et pas un autre modèle ?

On aurait pu utiliser une approche plus simple (TF-IDF + Logistic Regression).
Mais les décisions de justice ont un langage très spécifique, technique, avec
beaucoup de contexte implicite. CamemBERT, entraîné sur du français, comprend
ces nuances bien mieux qu'un simple comptage de mots.

**Ce qu'on va faire :**
1. Baseline rapide : TF-IDF pour avoir un point de comparaison
2. Modèle principal : CamemBERT fine-tuné sur nos décisions
3. Comparaison des deux approches

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Préparation des données
df_model = df_corpus[['texte', 'categorie', 'label']].dropna()
df_model['texte'] = df_model['texte'].astype(str)

# Split train/test : 80% entraînement, 20% évaluation
X_train, X_test, y_train, y_test = train_test_split(
    df_model['texte'],
    df_model['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_model['label']  # important : garde les proportions de classes
)

print(f'Train : {len(X_train)} decisions')
print(f'Test  : {len(X_test)} decisions')

In [ ]:
print('Entrainement du baseline TF-IDF...')

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=500, C=1.0)
lr.fit(X_train_tfidf, y_train)
y_pred_baseline = lr.predict(X_test_tfidf)

# On construit target_names uniquement depuis les labels présents dans y_test
# Evite l'erreur quand une catégorie du référentiel n'a aucune décision collectée
labels_presents = sorted(y_test.unique())
categories = [id2label[i] for i in labels_presents]

print('\nBaseline TF-IDF + Logistic Regression :')
print('=' * 60)
print(classification_report(y_test, y_pred_baseline,
                             labels=labels_presents,
                             target_names=categories))

In [ ]:
# ── MODELE PRINCIPAL : CamemBERT ──
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
import torch

MODEL_NAME = 'camembert-base'
NUM_LABELS = len(label2id)

# Tokenisation
print('Chargement du tokenizer CamemBERT...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=256  # decisions plus longues qu'Allocine -> 256 tokens
    )

# Construction des datasets HuggingFace
train_df = pd.DataFrame({'text': X_train.tolist(), 'label': y_train.tolist()})
test_df  = pd.DataFrame({'text': X_test.tolist(),  'label': y_test.tolist()})

train_dataset = Dataset.from_pandas(train_df).map(tokenize, batched=True)
test_dataset  = Dataset.from_pandas(test_df).map(tokenize,  batched=True)

print(f'Dataset train tokenise : {len(train_dataset)} exemples')
print(f'Dataset test tokenise  : {len(test_dataset)} exemples')

In [ ]:
# Chargement du modèle et configuration de l'entraînement
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./results_judilibre',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=20,
    warmup_ratio=0.1,        # évite les sauts brutaux au début
    weight_decay=0.01,       # régularisation contre le surapprentissage
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print('Entrainement CamemBERT lance...')
print('(Environ 10-15 min sur GPU Colab T4)')
trainer.train()
print('Entrainement termine !')

---
## Phase 4 — Evaluation et comparaison des modeles
---

### Pourquoi comparer les deux modèles ?

C'est ce qui distingue un projet sérieux d'un simple tutoriel.
On mesure objectivement si la complexité de CamemBERT est justifiée
par rapport au baseline simple.

In [ ]:
# Evaluation CamemBERT
predictions = trainer.predict(test_dataset)
y_pred_camembert = np.argmax(predictions.predictions, axis=1)
y_true = test_dataset['label']

print('CamemBERT fine-tune :')
print('=' * 60)
print(classification_report(y_true, y_pred_camembert, target_names=categories))

In [ ]:
# Visualisation comparative : Baseline vs CamemBERT
from sklearn.metrics import f1_score

f1_baseline   = f1_score(y_test, y_pred_baseline,   average='weighted')
f1_camembert  = f1_score(y_true, y_pred_camembert,  average='weighted')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Comparaison F1
modeles = ['TF-IDF\n+ LogReg', 'CamemBERT\nfine-tune']
scores  = [f1_baseline, f1_camembert]
colors_bar = ['#ADB5BD', '#2E86AB']
bars = axes[0].bar(modeles, scores, color=colors_bar, width=0.5)
axes[0].set_ylim(0, 1)
axes[0].set_title('F1-score weighted', fontsize=12, fontweight='bold')
axes[0].set_ylabel('F1-score')
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{score:.3f}', ha='center', fontweight='bold')

# Matrice de confusion Baseline
cm_b = confusion_matrix(y_test, y_pred_baseline)
ConfusionMatrixDisplay(cm_b, display_labels=[c[:12] for c in categories]).plot(
    ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion — Baseline', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

# Matrice de confusion CamemBERT
cm_c = confusion_matrix(y_true, y_pred_camembert)
ConfusionMatrixDisplay(cm_c, display_labels=[c[:12] for c in categories]).plot(
    ax=axes[2], cmap='Blues', colorbar=False)
axes[2].set_title('Confusion — CamemBERT', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Gain CamemBERT vs Baseline : +{(f1_camembert - f1_baseline)*100:.1f} points de F1')

---
## Phase 5 — Extraction d'entités nommées (NER) avec spaCy
---

### Pourquoi le NER après la classification ?

La classification dit *quel type* d'infraction. Le NER dit *qui, quoi, où, quand*.
Ensemble, ils constituent un pipeline complet d'analyse de documents juridiques.

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load('fr_core_news_md')

def extraire_entites(texte):
    """Extrait les entités nommées d'un texte juridique"""
    doc = nlp(texte[:1000])  # limite pour la vitesse
    entites = [(ent.text, ent.label_) for ent in doc.ents]
    return entites

# Application sur un echantillon
sample = df_corpus.groupby('categorie').first().reset_index()

print('Entites extraites par categorie :')
print('=' * 60)
for _, row in sample.iterrows():
    entites = extraire_entites(str(row['texte']))
    print(f'\nCategorie : {row["categorie"]}')
    print(f'Texte     : {str(row["texte"])[:150]}...')
    if entites:
        for texte_ent, type_ent in entites[:5]:
            print(f'  [{type_ent}] {texte_ent}')
    else:
        print('  (aucune entite detectee)')

In [ ]:
# Analyse globale des entités sur tout le corpus
print('Extraction des entites sur tout le corpus...')

toutes_entites = []
for texte in df_corpus['texte'].head(100):  # limite pour la vitesse
    entites = extraire_entites(str(texte))
    toutes_entites.extend(entites)

df_entites = pd.DataFrame(toutes_entites, columns=['texte', 'type'])

# Visualisation par type d'entité
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des types
type_counts = df_entites['type'].value_counts().head(10)
axes[0].bar(type_counts.index, type_counts.values, color='#2E86AB')
axes[0].set_title('Types d entites les plus frequents', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Entités les plus fréquentes
top_entites = df_entites['texte'].value_counts().head(15)
axes[1].barh(top_entites.index, top_entites.values, color='#2E86AB')
axes[1].set_title('Entites les plus mentionnees', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('ner_analyse.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTotal entites extraites : {len(df_entites)}')
print(df_entites['type'].value_counts())

---
## Phase 6 — Demonstration : pipeline complet sur une nouvelle decision
---

Preuve que le pipeline fonctionne de bout en bout sur
une nouvelle decision que le modele n'a jamais vue.

In [ ]:
def pipeline_complet(texte_decision):
    """
    Pipeline complet : texte -> categorie + entites
    C'est ce qu'on presente en entretien ou sur GitHub
    """
    print('ANALYSE DE LA DECISION')
    print('=' * 60)
    print(f'Texte : {texte_decision[:200]}...')
    print()

    # 1. Classification
    inputs = tokenizer(
        texte_decision,
        return_tensors='pt',
        truncation=True,
        max_length=256
    )
    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_id = torch.argmax(probs).item()
    pred_label = id2label[pred_id]
    confidence = probs[pred_id].item()

    print(f'CLASSIFICATION')
    print(f'  Categorie predite : {pred_label}')
    print(f'  Confiance         : {confidence:.1%}')
    print()

    # Top 3 categories
    print('  Top 3 categories :')
    top3 = torch.topk(probs, 3)
    for score, idx in zip(top3.values, top3.indices):
        print(f'    {id2label[idx.item()]:<40} {score.item():.1%}')
    print()

    # 2. NER
    entites = extraire_entites(texte_decision)
    print('ENTITES EXTRAITES')
    if entites:
        for texte_e, type_e in entites:
            print(f'  [{type_e}] {texte_e}')
    else:
        print('  Aucune entite detectee')

    return pred_label, confidence, entites


# Test sur des exemples de ton choix
exemples = [
    """
    La cour constate que le prévenu a procédé au détournement de fonds publics
    à hauteur de 50 000 euros, en falsifiant les documents comptables de la société
    entre janvier et mars 2022. Les faits constituent une escroquerie aggravée.
    """,
    """
    Le tribunal retient que l'accusé conduisait sous l'empire d'un état alcoolique
    avec un taux de 1.8 gramme par litre de sang, causant un accident de la circulation
    ayant entraîné des blessures graves sur la victime.
    """,
    """
    Il est reproché au prévenu d'avoir importé des marchandises de contrebande
    en dissimulant leur nature et leur valeur réelle lors du passage en douane,
    causant un préjudice fiscal à l'État de 120 000 euros.
    """
]

for i, exemple in enumerate(exemples, 1):
    print(f'\n--- EXEMPLE {i} ---')
    pipeline_complet(exemple.strip())
    print()

---
## Bilan du projet

- Collecte de donnees reelles via une API gouvernementale (Judilibre)
- Construction d'un referentiel de labels depuis l'ONDRP
- Pipeline NLP complet : tokenisation, fine-tuning CamemBERT, evaluation
- Extraction d'entites nommees avec spaCy
- Comparaison objective baseline vs modele avance
- Demo end-to-end sur de nouvelles decisions